# Scale Output

This notebook scales the SLP, U850, and V850 variables in the finetuned Prithvi-WxC output. The variables after the anlysis time erroneously weren't scaled back to physical units. This notebook fixes this.

In [1]:
from pathlib import Path

from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr

## Load Scaling Factors

In [18]:
from prithvi_precip.data.merra2 import SURFACE_VARS, VERTICAL_VARS, LEVELS
from prithvi_precip.utils import load_and_interp_climatology
from PrithviWxC.dataloaders.merra2 import output_scalers

var_inds = [9, 17, 18, -18, -4]

climatology_path = Path("/mnt/ssd-data2/TEMP/hf_cache/")
output_sig = output_scalers(
    SURFACE_VARS,
    VERTICAL_VARS,
    LEVELS,
    str(climatology_path / "climatology/anomaly_variance_surface.nc"),
    str(climatology_path / "climatology/anomaly_variance_vertical.nc"),
)[..., None, None] ** 0.5

output_sig_slp = output_sig[9].numpy()
output_sig_u850 = output_sig[-18].numpy()
output_sig_v850 = output_sig[-4].numpy()

## Rescale SLP, U850, and V850 after analysis time

In [46]:
results_path = Path("/mnt/ssd-data1/prithvi/data/results_v0.2_np3v/")
# Uncomment the lines below to create new directory for output
#output_path = Path("/mnt/ssd-data1/prithvi/data/scaled/results_v0.1_np2v")
#output_path.mkdir(exist_ok=True, parents=True)

results = sorted(list(results_path.glob("**/*.nc")))
for path in tqdm(results):
    data = xr.load_dataset(path)
    new_data = data.copy(deep=True)
    for time_ind in range(1, new_data.valid_time.size):
        valid_time = new_data.valid_time.data[time_ind]
        clim = load_and_interp_climatology(valid_time, climatology_path)[:, :-1]
        clim_slp = clim[9]
        clim_u850 = clim[-18]
        clim_v850 = clim[-4]

        new_data["slp"].data[time_ind] =  new_data["slp"].data[time_ind] * output_sig_slp + clim_slp
        new_data["u850"].data[time_ind] =  new_data["u850"].data[time_ind] * output_sig_u850 + clim_u850
        new_data["v850"].data[time_ind] =  new_data["v850"].data[time_ind] * output_sig_v850 + clim_v850

    # Uncomment this to write results
    output_file = output_path / path.name
    #new_data.to_netcdf(output_file)

  0%|                                                                                                    | 1/988 [00:21<5:57:18, 21.72s/it]


KeyboardInterrupt: 